
Selección $\rightarrow$ Limpieza $\rightarrow$ Transformación $\rightarrow$ Minería $\rightarrow$ Evaluación $\rightarrow$ Presentación


## 1. Selección de Datos (Data Selection)
Este bloque cumple la función crucial de Selección y Carga de Datos (Data Selection):

1. Inspección de archivos y mapeo dinámico (encontrar_archivo_cercano): En lugar de depender de rutas rígidas que podrían fallar si los archivos cambian ligeramente de nombre en el entorno de Colab, el código busca coincidencias clave (como 'viajeros-hospedados', 'gasto_promedio') para garantizar una carga robusta y evitar errores de tipo FileNotFoundError.
2. Filtro inteligente de entrada: Tras el proceso de depuración, este bloque carga únicamente los 4 datasets esenciales (df_gasto, df_hospedados, df_vuelos, y df_bcra) que aportan valor analítico directo al pipeline. Al excluir los datasets huérfanos que no se utilizaban, se optimiza significativamente el uso de memoria RAM y la agilidad de procesamiento del entorno.
3. Verificación de dimensiones: Finaliza imprimiendo las formas (shape) de las tablas para tener una confirmación empírica inmediata de que los datos brutos se han levantado correctamente con el volumen esperado.

Este primer bloque es el cimiento de dicha hipótesis porque reúne y disponibiliza las piezas fundamentales del rompecabezas:

1. Carga la variable a predecir (la demanda hotelera o viajeros de df_hospedados).
2. Carga los predictores de conectividad de transporte (pasajeros, asientos y vuelos de df_vuelos).
3. Carga los estimadores de comportamiento de consumo y contexto macroeconómico (gasto_promedio de df_gasto y el tipo de cambio del df_bcra para dolarizar los gastos).

In [1]:
import numpy as np
import pandas as pd
import os

archivos_presentes = os.listdir('.')

def encontrar_archivo_cercano(patron_nombre):
    for archivo in archivos_presentes:
        if patron_nombre.lower() in archivo.lower():
            return archivo
    return patron_nombre

file_gasto = encontrar_archivo_cercano('gasto_promedio')
file_hospedados = encontrar_archivo_cercano('viajeros-hospedados')
file_vuelos = encontrar_archivo_cercano('vuelos_asientos')
file_bcra = encontrar_archivo_cercano('tipos-de-cambio')

df_gasto = pd.read_csv(file_gasto)
df_hospedados = pd.read_csv(file_hospedados)
df_vuelos = pd.read_csv(file_vuelos)
df_bcra = pd.read_csv(file_bcra)

print("Dimensiones iniciales cargadas exitosamente:")
print(f"- Hospedados: {df_hospedados.shape} ({file_hospedados})")
print(f"- Gasto promedio: {df_gasto.shape} ({file_gasto})")
print(f"- Vuelos/Pasajeros: {df_vuelos.shape} ({file_vuelos})")
print(f"- Tipo de Cambio BCRA: {df_bcra.shape} ({file_bcra})")

Dimensiones iniciales cargadas exitosamente:
- Hospedados: (3178, 5) (viajeros-hospedados-residentes-y-no-residentes-por-destino.csv)
- Gasto promedio: (400, 3) (tur_int_turistas_residentes_gasto_promedio_trim_destino_serie.csv)
- Vuelos/Pasajeros: (6877, 5) (vuelos_asientos_pasajeros.csv)
- Tipo de Cambio BCRA: (20727, 12) (tipos-de-cambio-historicos.csv)


## 2. Limpieza y Preparación de Datos (Data Cleaning & Preparation)
Este bloque realiza la sanitización y adecuación estructural de las variables recolectadas:

### 1. Justificación del Paso: ¿Por qué hace lo que hace con los datos?
* **Estandarización de nombres de columnas:** Al aplicar `.str.strip().str.lower()`, eliminamos espacios ocultos y homogeneizamos las mayúsculas/minúsculas. Esto previene errores de sintaxis comunes al referenciar variables y facilita la automatización del pipeline.
* **Normalización de entidades geográficas (`region_destino`):** En las fuentes originales, un mismo destino puede figurar de múltiples maneras (ej. *'Pba - Partidos Del Gba'* vs *'Buenos Aires'*). El uso del diccionario `mapeo_regiones` consolida estas discrepancias en categorías uniformes. Sin esta unificación, el análisis regional se segmentaría artificialmente en grupos duplicados.
* **Tratamiento temporal y parseo estricto (`pd.to_datetime`):** Las marcas de tiempo de las encuestas y registros hoteleros se convierten a formato de fecha con manejo de errores coercitivo (`errors='coerce'`) para aislar nulos, y luego se transforman al período mensual (`Period('M')`). Esto es indispensable para sincronizar la periodicidad mensual fija que demanda el análisis temporal.

### 2. Relación con la Hipótesis del Proyecto
La hipótesis plantea que la afluencia turística regional está influenciada por factores temporales específicos (estacionalidad) y perfiles económicos regionales.
* Esta celda **hace posible la integración multidimensional**. Al unificar estrictamente bajo una misma clave geográfica (`region_destino`) y temporal (`mes`), los datasets de gasto hotelero y volumen de viajeros de cabotaje quedan listos para acoplarse físicamente en un solo plano analítico.
* Garantiza que la correlación espacial sea válida; de lo contrario, estaríamos intentando relacionar el gasto promedio de un destino con la afluencia turística de otra entidad registrada bajo un nombre ligeramente distinto.

In [2]:
import pandas as pd

datasets = [
    df_gasto,
    df_hospedados,
    df_vuelos,
    df_bcra
]

for df in datasets:
    df.columns = df.columns.str.strip().str.lower()

if 'region_de_destino' in df_hospedados.columns:
    df_hospedados.rename(columns={'region_de_destino': 'region_destino'}, inplace=True)

mapeo_regiones = {
    'Pba - Partidos Del Gba': 'Buenos Aires',
    'Pba - Resto': 'Buenos Aires',
    'Buenos Aires': 'Buenos Aires',
    'Caba': 'Caba',
    'Cuyo': 'Cuyo',
    'Córdoba': 'Córdoba',
    'Litoral': 'Litoral',
    'Norte': 'Norte',
    'Patagonia': 'Patagonia'
}

df_gasto['region_destino'] = df_gasto['region_destino'].astype(str).str.strip().str.title()
df_gasto['region_destino'] = df_gasto['region_destino'].map(mapeo_regiones).fillna(df_gasto['region_destino'])

df_hospedados['region_destino'] = df_hospedados['region_destino'].astype(str).str.strip().str.title()
df_hospedados['region_destino'] = df_hospedados['region_destino'].map(mapeo_regiones).fillna(df_hospedados['region_destino'])

df_hospedados['mes'] = pd.to_datetime(df_hospedados['indice_tiempo'], errors='coerce').dt.to_period('M')
df_gasto['mes'] = pd.to_datetime(df_gasto['indice_tiempo'], errors='coerce').dt.to_period('M')

df_hosp_m = df_hospedados.groupby(['mes', 'region_destino'])['viajeros'].sum().reset_index()
df_gasto_m = df_gasto.groupby(['mes', 'region_destino'])['gasto_promedio'].mean().reset_index()

print("Regiones únicas mapeadas en Hospedados:", sorted(df_hosp_m['region_destino'].unique()))
print("Regiones únicas mapeadas en Gasto:", sorted(df_gasto_m['region_destino'].unique()))
print(f"Rango temporal de análisis: {df_hosp_m['mes'].min()} hasta {df_hosp_m['mes'].max()}")

Regiones únicas mapeadas en Hospedados: ['Buenos Aires', 'Caba', 'Cuyo', 'Córdoba', 'Litoral', 'Norte', 'Patagonia']
Regiones únicas mapeadas en Gasto: ['Buenos Aires', 'Caba', 'Cuyo', 'Córdoba', 'Litoral', 'Norte', 'Patagonia']
Rango temporal de análisis: 2007-01 hasta 2025-11


## 3. Transformación de Datos (Data Transformation)
En esta fase se construye la estructura analítica consolidada (Data Mart) mediante técnicas de ingeniería de variables e integración multidimensional:

### 1. Justificación del Paso: ¿Por qué hace lo que hace con los datos?
* **Alineación de Frecuencia Temporal (Mensual):** El comportamiento de los viajeros y vuelos opera de forma natural a nivel mensual, mientras que la encuesta de gasto promedio del Ministerio de Turismo se publica trimestralmente. El pipeline realiza un mapeo cruzado descendente (*downsampling*), propagando el gasto medio trimestral a sus tres meses integrantes para no perder granularidad en el análisis fino.
* **Filtro de Conectividad Nacional:** Se filtra estrictamente la columna `clasificacion_vuelo` para procesar únicamente vuelos de **Cabotaje**, alineándose de forma consistente con el objetivo del estudio.
* **Anotación sobre el Rango Temporal Efectivo:** Dado que los registros de conectividad aérea de la ANAC comienzan en enero de 2017, la fusión interna de datos provoca la pérdida de las observaciones de 2012-2016. De esta forma, el análisis final consolidado se focaliza de manera realista y completa en el periodo 2017-2024.
* **Corrección Monetaria (Dolarización del Gasto):** Argentina experimenta alta inflación. Trabajar con gastos nominales históricos en pesos desvirtuaría las tendencias reales. Al dividir el gasto promedio por el tipo de cambio oficial de referencia del BCRA (`dolar_referencia_com_3500`), el científico de datos estabiliza la escala a dólares estadounidenses (USD), permitiendo comparar de forma limpia el poder adquisitivo real a lo largo de más de una década.

### 2. Relación con la Hipótesis del Proyecto
* **La Hipótesis de la Conectividad y Gasto Estabilizado:** La hipótesis propone un nexo lógico y estable entre el flujo aéreo de pasajeros y el consumo local. Sin la conversión a moneda fuerte (USD) en este paso de transformación, las correlaciones resultantes estarían fuertemente sesgadas por la devaluación monetaria en pesos, invalidando la robustez de los algoritmos predictivos.

In [3]:
df_hospedados['trimestre'] = pd.to_datetime(
    df_hospedados['indice_tiempo']
).dt.to_period('Q')
df_gasto['trimestre'] = pd.to_datetime(
    df_gasto['indice_tiempo']
).dt.to_period('Q')

df_hosp_m = (
    df_hospedados.groupby(['mes', 'trimestre', 'region_destino'])['viajeros']
    .sum()
    .reset_index()
)
df_gasto_q = (
    df_gasto.groupby(['trimestre', 'region_destino'])['gasto_promedio']
    .mean()
    .reset_index()
)

df_vuelos['mes'] = pd.to_datetime(df_vuelos['indice_tiempo']).dt.to_period('M')
df_vuelos_cabotaje = df_vuelos[df_vuelos['clasificacion_vuelo'].str.lower() == 'cabotaje'].copy()

df_vuelos_m = (
    df_vuelos_cabotaje.groupby('mes')[['pasajeros', 'asientos', 'vuelos']]
    .sum()
    .reset_index()
)

df_bcra['mes'] = pd.to_datetime(df_bcra['indice_tiempo']).dt.to_period('M')
col_dolar = 'dolar_referencia_com_3500'
if col_dolar not in df_bcra.columns:
    opciones = [c for c in df_bcra.columns if '3500' in c or 'oficial' in c or 'tipo_unico' in c]
    col_dolar = opciones[0] if opciones else df_bcra.columns[1]

df_bcra_m = (
    df_bcra.groupby('mes')[col_dolar]
    .mean()
    .reset_index()
)
df_bcra_m.rename(columns={col_dolar: 'dolar_referencia_com_3500'}, inplace=True)

df_consolidado = pd.merge(
    df_hosp_m, df_gasto_q, on=['trimestre', 'region_destino'], how='left'
)
df_consolidado = pd.merge(df_consolidado, df_vuelos_m, on='mes', how='left')
df_consolidado = pd.merge(df_consolidado, df_bcra_m, on='mes', how='left')

df_depurado = df_consolidado.dropna(subset=['viajeros', 'gasto_promedio', 'pasajeros']).copy()

anio = df_depurado['mes'].dt.year
df_depurado = df_depurado[~anio.isin([2020, 2021])].copy()

df_depurado['mes_str'] = df_depurado['mes'].astype(str)
df_depurado['numero_mes'] = df_depurado['mes'].dt.month
df_depurado['gasto_promedio_usd'] = df_depurado['gasto_promedio'] / df_depurado['dolar_referencia_com_3500']

print(f"Data Mart consolidado con éxito. Registros resultantes: {len(df_depurado)}")

Data Mart consolidado con éxito. Registros resultantes: 462


## 4. Análisis Exploratorio de Datos - EDA
Esta etapa de exploración visual y estadística permite comprender de forma empírica la dinámica interna del Data Mart consolidado:

### 1. Justificación del Paso y Decisiones Críticas de Negocio
* **Evolución Temporal por Región:** El gráfico de líneas revela patrones dinámicos claros. Es fundamental para identificar picos estacionales de turismo de invierno (ej. Patagonia en Julio) y de verano (ej. Costa Atlántica en Enero), lo que demuestra que un solo modelo generalizado para todo el país perdería la precisión local.
* **Dispersión y Correlaciones Multivariables:** Evalúa la relación directa entre el tráfico aéreo de la ANAC y los huéspedes reportados en la EOH. Permite ver que, a mayor tráfico aéreo, mayor es la tasa de huéspedes en las regiones de destino masivo.

### 2. Decisión del Científico de Datos: Exclusión Estratégica del Periodo de Pandemia (2020-2021)
* **El Desafío de los Eventos Extremos:** El confinamiento estricto decretado por el COVID-19 provocó el cese de vuelos comerciales de cabotaje y el cierre preventivo de la hotelería argentina durante gran parte de 2020 y 2021.
* **Por qué eliminar este periodo:** Desde una perspectiva estrictamente estadística, estos meses actúan como *outliers extremos de choque estructural*. Mantener el bloque 2020-2021 en el conjunto de entrenamiento de los algoritmos degradaría severamente el desempeño de predicción de la demanda regular futura. Al limpiar la serie eliminando este sesgo sistémico artificial, nos quedamos con comportamientos turísticos orgánicos que responden a la oferta, demanda, y estacionalidad estándar, garantizando la viabilidad del modelo predictivo.

In [4]:
import plotly.express as px

fig_estacionalidad = px.line(
    df_depurado,
    x='mes_str',
    y='viajeros',
    color='region_destino',
    title='Evolución Mensual de Viajeros Hospedados por Región (EOH)',
    labels={
        'mes_str': 'Período Mensual',
        'viajeros': 'Cantidad de Viajeros',
        'region_destino': 'Región',
    },
)
fig_estacionalidad.show()

fig_scatter = px.scatter(
    df_depurado,
    x='pasajeros',
    y='viajeros',
    color='region_destino',
    size='gasto_promedio_usd',
    hover_data=['mes_str'],
    title=(
        'Conectividad Aérea (ANAC) vs. Ocupación Hotelera (Tamaño = Gasto USD)'
    ),
    labels={
        'pasajeros': 'Pasajeros Aéreos de Cabotaje',
        'viajeros': 'Viajeros Hospedados',
        'region_destino': 'Región',
        'gasto_promedio_usd': 'Gasto Promedio (USD)',
    },
)
fig_scatter.show()

df_pivot = df_depurado.pivot_table(
    index='region_destino',
    columns='numero_mes',
    values='viajeros',
    aggfunc='mean',
)

nombres_meses = {
    1: 'Ene',
    2: 'Feb',
    3: 'Mar',
    4: 'Abr',
    5: 'May',
    6: 'Jun',
    7: 'Jul',
    8: 'Ago',
    9: 'Sep',
    10: 'Oct',
    11: 'Nov',
    12: 'Dic',
}
meses_presentes = [
    nombres_meses[col] for col in df_pivot.columns if col in nombres_meses
]

fig_heatmap = px.imshow(
    df_pivot,
    labels=dict(x='Mes del Año', y='Región', color='Promedio Viajeros'),
    x=meses_presentes,
    title='Mapa de Calor: Concentración Ocupacional Mensual por Región',
    color_continuous_scale='YlOrRd',
)
fig_heatmap.show()

cols_corr = [
    c
    for c in [
        'viajeros',
        'pasajeros',
        'gasto_promedio_usd',
        'dolar_referencia_com_3500',
    ]
    if c in df_depurado.columns
]

fig_corr = px.imshow(
    df_depurado[cols_corr].corr(),
    text_auto=True,
    title='Matriz de Correlación de Variables del Data Mart',
    color_continuous_scale='Blues',
)
fig_corr.show()

## 4.1 Conclusiones y Hallazgos Clave del EDA (Corregidos)

### 📈 Evolución Mensual de Viajeros Hospedados por Región
* **Estacionalidad Regionalizada Marcada:** Se identifican dos tipos claros de comportamiento estacional. Las regiones como **Patagonia** exhiben un comportamiento bimodal con picos muy pronunciados en la temporada invernal (julio/agosto) y estival (enero/febrero). En contraste, la región de **CABA** muestra una estabilidad mucho más sólida y homogénea a lo largo del año debido a su perfil de turismo corporativo y cultural.
* **Tendencia Reciente de Estabilización:** Al observar la tendencia de largo plazo (excluyendo la pandemia), se observa una recuperación progresiva hasta alcanzar una meseta estable, con ciertas desaceleraciones en el transcurso de 2024 que reflejan la maduración o reconfiguración del mercado turístico interno ante el contexto económico actual.

### ✈️ Conectividad Aérea (ANAC) vs Ocupación Hotelera
* **Nivel de Asociación Lineal en Matriz:** El coeficiente de correlación entre los **viajeros hospedados** (aperturados a nivel regional) y los **pasajeros de cabotaje** nacionales se sitúa en un valor moderado de **0.17**. Esto indica que aunque la conectividad aérea nacional general facilita el flujo global de personas, los viajeros de cada región particular responden fuertemente a dinámicas locales y estacionales propias que van más allá del volumen aéreo agregado del país.
* **El Rol del Gasto Promedio (Tamaño de la Burbuja):** Los destinos con mayor gasto promedio en dólares (como la Patagonia) no siempre representan el mayor volumen absoluto de pasajeros, pero sí el mayor impacto económico por turista. Esto justifica la necesidad de utilizar modelos que incorporen tanto variables de volumen físico (pasajeros) como variables de valor económico (gasto dolarizado) para caracterizar adecuadamente los perfiles de mercado.

### 🌡️ Mapa de Calor: Concentración Ocupacional Mensual por Región
* **Identificación de la Temporada Alta de Verano:** El mapa de calor ilustra una concentración crítica de ocupación hotelera en los meses de **Enero y Febrero** en la provincia de **Buenos Aires** agregada (impulsada significativamente por el turismo masivo de la Costa Atlántica) y **Córdoba**, donde el color se intensifica.
* **Valles de Demanda Coincidentes:** Los valles reales y sistemáticos de demanda a nivel nacional se sitúan en **Mayo y Junio** (previos a las vacaciones de invierno), mostrando los colores más fríos en la gran mayoría de las regiones. Los meses de septiembre y octubre muestran niveles de actividad media-alta en destinos específicos como la Patagonia, CABA y el Norte.

## 5. Técnica de Minería y Modelos (KDD & Machine Learning)
Esta fase aplica metodologías avanzadas de minería de datos estructuradas para el modelado matemático del comportamiento del mercado turístico:

### 1. Justificación del Paso y Decisiones Técnicas
* **Segmentación No Supervisada Híbrida (K-Means):** En lugar de tratar a todas las regiones de destino bajo el mismo criterio de volumen o solo por su nombre geográfico, el algoritmo K-Means identifica patrones latentes agrupando el volumen de viajeros con su nivel de gasto medio en USD. Esto segmenta al país de forma inteligente (ej. destinos de alto volumen/bajo gasto, destinos premium de bajo volumen/alto gasto, etc.). La etiqueta del clúster resultante se reinyecta al modelo supervisado para enriquecer sus características de entrada.
* **Codificación Cíclica de la Estacionalidad:** La variable del mes representada simplemente como un entero (1 al 12) introduce un problema lógico: el mes 12 (Diciembre) y el mes 1 (Enero) están extremadamente distantes numéricamente, a pesar de ser adyacentes en el tiempo real. Para resolver esto de manera rigurosa, se transforman los meses en coordenadas circulares trigonométricas usando **Seno (`mes_sin`) y Coseno (`mes_cos`)**, preservando la continuidad del ciclo temporal anual en el modelo predictivo.
* **Estrategia de Validación Cronológica Estricta (80/20 sin Data Leakage):** Al trabajar con series de tiempo, la división aleatoria clásica (`train_test_split`) está prohibida porque causaría filtración de información del futuro hacia el pasado (*data leakage*). En su lugar, se implementa una **partición temporal estricta**: entrena con el pasado histórico (hasta Diciembre de 2022) y valida las predicciones con el futuro no visto (desde Enero de 2023 en adelante).

### 2. Explicación de los Modelos Entrenados
En este pipeline se implementa una arquitectura híbrida secuencial que combina aprendizaje no supervisado y supervisado:
1. **Modelo de Segmentación (K-Means):**
   * **Objetivo:** Agrupar los registros en 3 perfiles turísticos basados en dos dimensiones clave estandarizadas: el volumen físico de `viajeros` y el valor económico medido por el `gasto_promedio_usd`.
   * **Resultados de los Clústeres:**
     * *Clúster 0 (Perfil Consolidado de Alto Volumen):* Destinos masivos con alta afluencia y gasto moderado.
     * *Clúster 1 (Perfil Emergente/Bajo Impacto):* Destinos con menor flujo de visitantes y gasto promedio más bajo.
     * *Clúster 2 (Perfil Premium):* Destinos de alta gama caracterizados por un volumen moderado-alto de viajeros pero con el mayor gasto promedio en dólares por turista.
2. **Modelo Predictivo (Gradient Boosting Regressor):**
   * **Objetivo:** Predecir cuantitativamente la afluencia de viajeros futuros por región de destino.
   * **Funcionamiento:** Es un modelo de ensamble basado en árboles de decisión que se construyen de forma secuencial. Cada nuevo árbol corrige los errores residuales de los árboles anteriores mediante el descenso de gradiente. Al utilizar parámetros optimizados (`n_estimators=250`, `learning_rate=0.04`, `max_depth=5`), el algoritmo captura relaciones complejas y no lineales entre las variables de conectividad aérea de la ANAC, la estacionalidad cíclica y las características geográficas de las regiones.

### 3. Relación con la Hipótesis del Proyecto
* **Validación Estadística del Ensamble:** El uso de un regresor no lineal potente como **Gradient Boosting Regressor** que asocia la conectividad aérea, las variaciones macroeconómicas y los componentes estacionales cíclicos, permite alcanzar un **R² del 87%**. Esto valida cuantitativamente la hipótesis: la dinámica conjunta de conectividad de cabotaje, estacionalidad temporal e ingresos es un excelente indicador de la afluencia turística real del país.

In [5]:
import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

df_depurado_sin_nans = df_depurado.dropna(
    subset=[
        'viajeros',
        'pasajeros',
        'asientos',
        'vuelos',
        'gasto_promedio_usd',
    ]
).copy()

features_cluster = df_depurado_sin_nans[['viajeros', 'gasto_promedio_usd']]
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_cluster)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_depurado_sin_nans['cluster_perfil'] = kmeans.fit_predict(features_scaled)

print('=== PERFILES TURÍSTICOS (K-Means) ===')
print(
    df_depurado_sin_nans.groupby('cluster_perfil')[
        ['viajeros', 'gasto_promedio_usd']
    ]
    .mean()
    .round(2)
)

df_depurado_sin_nans['mes_sin'] = np.sin(
    2 * np.pi * df_depurado_sin_nans['numero_mes'] / 12
)
df_depurado_sin_nans['mes_cos'] = np.cos(
    2 * np.pi * df_depurado_sin_nans['numero_mes'] / 12
)

df_model = pd.get_dummies(
    df_depurado_sin_nans, columns=['region_destino', 'cluster_perfil'], drop_first=True, dtype=int
)

columnas_no_predictoras = [
    'mes',
    'mes_str',
    'trimestre',
    'mes_fecha',
    'viajeros',
    'gasto_promedio',
    'gasto_promedio_usd',
    'indice_tiempo',
]

col_dolar = 'dolar_referencia_com_3500'
if col_dolar in df_model.columns:
  columnas_no_predictoras.append(col_dolar)

features_model = [
    c
    for c in df_model.columns
    if c not in columnas_no_predictoras
    and pd.api.types.is_numeric_dtype(df_model[c])
]

X = df_model[features_model]
y = df_model['viajeros']

fechas = pd.to_datetime(df_model['mes'].astype(str))
orden = np.argsort(fechas.to_numpy())

X = X.iloc[orden].reset_index(drop=True)
y = y.iloc[orden].reset_index(drop=True)
fechas = fechas.iloc[orden].reset_index(drop=True)

filtro_train = fechas < '2023-01-01'

X_train = X[filtro_train].reset_index(drop=True)
X_test = X[~filtro_train].reset_index(drop=True)
y_train = y[filtro_train].reset_index(drop=True)
y_test = y[~filtro_train].reset_index(drop=True)

print('\n=== EVALUACIÓN CRONOLÓGICA ===')
print('Último período de entrenamiento:', fechas[filtro_train].max())
print('Primer período de prueba:', fechas[~filtro_train].min())

model_gbr = GradientBoostingRegressor(
    n_estimators=250,
    learning_rate=0.04,
    max_depth=5,
    min_samples_split=4,
    subsample=0.85,
    random_state=42,
)

model_gbr.fit(X_train, y_train)

y_pred = model_gbr.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('\n=== EVALUACIÓN DEL MODELO PREDICTIVO OPTIMIZADO ===')
print(f'Registros entrenamiento: {len(X_train)} | Prueba: {len(X_test)}')
print(f'Error Absoluto Medio (MAE): {mae:,.2f} viajeros')
print(f'Coeficiente de Determinación (R²): {r2:.4f}')

joblib.dump(model_gbr, 'modelo_predictivo_turismo.pkl')
joblib.dump(features_model, 'columnas_modelo.pkl')
joblib.dump(
    {
        'scaler_cluster': scaler,
        'kmeans': kmeans,
        'columnas_modelo': features_model,
    },
    'preprocesamiento_turismo.pkl',
)
df_depurado_sin_nans.to_csv(
    'dataset_turismo_consolidado_mensual.csv', index=False
)

=== PERFILES TURÍSTICOS (K-Means) ===
                 viajeros  gasto_promedio_usd
cluster_perfil                               
0               359293.26              192.15
1               182911.64              123.29
2               305653.67              345.66

=== EVALUACIÓN CRONOLÓGICA ===
Último período de entrenamiento: 2022-12-01 00:00:00
Primer período de prueba: 2023-01-01 00:00:00

=== EVALUACIÓN DEL MODELO PREDICTIVO OPTIMIZADO ===
Registros entrenamiento: 336 | Prueba: 126
Error Absoluto Medio (MAE): 27,297.15 viajeros
Coeficiente de Determinación (R²): 0.8729


---
## 6. Conclusión y Valor de Negocio
- **Validación de la Hipótesis:** La combinación de la serie histórica filtrada de vuelos de cabotaje (ANAC), estacionalidad mensual y perfiles de gasto reinyectados de K-Means permitió predecir la afluencia turística regional de manera muy robusta.
- **Impacto Comercial:** Al contar con un pipeline que captura la estacionalidad regional y la segmentación del perfil del turista, los operadores pueden predecir comportamientos futuros basándose en las proyecciones de conectividad aérea, optimizando sus estrategias comerciales, de inventario y tarifas dinámicas.